<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 4 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b && ollama pull embeddinggemma &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [1]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B (no native thinking or tool calling)
llm = LLM(model="gemma3:12b")

If you want to use another LLM, here are a couple of options that use the `OpenAI` library:

In [2]:
# from openai import OpenAI
# from illustrated_agents.llm import OpenAIClientLLM

# # Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma-3-12B-it-Q4_K_M", client=client)

# # LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma-3-12B-it", client=client)

# # Google's Gemini
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = OpenAIClientLLM(model="gemini-2.5-flash", client=client)

## 2 - Adding Retrieval Augmented Generation (RAG)


In the previous notebook, we added several forms of short-term memory (trimming and summarization). In this notebook we will cover Retrieval Augemented Generation (RAG) as a form of long-term memory.

We start by creating the EmbeddingModel class which uses the OpenAI-endpoint to query a given model. In our example, we are going to be using [EmbeddingGemma](https://ai.google.dev/gemma/docs/embeddinggemma) an embedding model with 308 million parameters. Quite a bit smaller than the 12 billion parameter LLM we have been using thus far!


In [6]:
import json
import urllib.request

class EmbeddingModel:
    """Generate embeddings."""

    def __init__(
        self,
        model: str,
        base_url: str = "http://localhost:11434/v1",
    ):
        """Initialize the embedding model with the given model."""
        self.model = model
        self.base_url = base_url

    def embed(self, text: str) -> list[float]:
        """Convert text into a numerical vector."""
        # POST to the OpenAI-compatible /embeddings endpoint
        request = urllib.request.Request(
            f"{self.base_url}/embeddings",
            data=json.dumps({"model": self.model, "input": text}).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(request) as resp:
            response = json.loads(resp.read())

        # Extract and return the embedding
        return response["data"][0]["embedding"]

We then initialize this class and we can use the same client since we are going to be using Ollama as our backend. 

In [7]:
# Initialize EmbeddingGemma
embedding_model = EmbeddingModel(model="embeddinggemma")

# Test the embedding model
output = embedding_model.embed("Dolphins are amazing!")
output

[-0.21747558,
 -0.01567787,
 0.03601467,
 -0.0049857525,
 -0.042537987,
 -0.038803127,
 -0.027158154,
 0.035217084,
 0.039407827,
 -0.028296778,
 -0.021544144,
 -0.038480375,
 -0.037420757,
 -0.013370458,
 0.061243117,
 0.031880256,
 0.015579869,
 -0.07982549,
 -0.04060092,
 -0.025447948,
 0.041066926,
 0.029795805,
 -0.008624076,
 -0.00090551743,
 0.05152579,
 0.011594693,
 -0.028225064,
 0.047016077,
 0.0021305713,
 0.02506623,
 0.0002318465,
 -0.001770541,
 0.013356723,
 0.008962886,
 -0.0029184285,
 0.029676717,
 -0.059625868,
 -0.032111786,
 0.091392994,
 -0.024812458,
 -0.034591235,
 0.05804671,
 -0.016386192,
 -0.019781549,
 -0.04010048,
 0.02328608,
 -0.020690933,
 0.04281413,
 -0.008274814,
 -0.0058891643,
 -0.0041595274,
 -0.0612561,
 -0.05579558,
 0.009133458,
 -0.020362759,
 -0.036887538,
 -0.015211727,
 0.009475907,
 -0.037046272,
 0.036663566,
 -0.019490872,
 -0.059709307,
 -0.009256026,
 0.0049812174,
 0.048680536,
 -0.061508756,
 0.002452944,
 -0.0051394533,
 -9.988251e

The output is a list of 768 values, each between -1 and 1.

We can use these values to compare different documents and calculate their similarity. This is typically calculated as the cosine similarity, which represents the angle between embeddings. A smaller angle means a higher similarity. The cosine similarity is calculated through the dot product of the embeddings and then divided by the product of their lengths for normalization.

Let's try it out!

In [8]:
from rich import print

# Create embeddings
embedding_a = embedding_model.embed("I love flamingos.")
embedding_b = embedding_model.embed("Dolphins use echolocation.")
embedding_c = embedding_model.embed("Flamingos are pink birds.")

# Calculate cosine similarity between A and B
dot_ab = sum(x * y for x, y in zip(embedding_a, embedding_b))
norm_a = sum(x * x for x in embedding_a) ** 0.5
norm_b = sum(x * x for x in embedding_b) ** 0.5
similarity_ab = dot_ab / (norm_a * norm_b)

# Calculate cosine similarity between A and C
dot_ac = sum(x * y for x, y in zip(embedding_a, embedding_c))
norm_c = sum(x * x for x in embedding_c) ** 0.5
similarity_ac = dot_ac / (norm_a * norm_c)

print(f"Similarity between A and B: {similarity_ab}")
print(f"Similarity between A and C: {similarity_ac}")

Similarity between A and B: 0.35467792870205467

Similarity between A and C: 0.6386974945250551






As such, the `RAGMemory` that we are going to implement has the following steps:

1) Embed all external documents the Agent has no direct access to (`__init__`)
2) Embed the user's query (`embedding_model.embed(query)`)
3) Compared the embeddings and create a similarity matrix (`.`)
4) Return the documents with the highest similarity
5) Add those documents to the prompt

In [9]:
from illustrated_agents.chapters.ch4 import Memory

class RAGMemory(Memory):
    """Long-term memory with RAG."""

    def __init__(self, embedding_model: EmbeddingModel, documents: list[str]):
        super().__init__()
        self.embedding_model = embedding_model
        self.documents = documents
        self.embeddings = [embedding_model.embed(doc) for doc in documents]

    def add(self, role: str, content: str):
        # Augment user queries with retrieved context before storing
        if role == "user":
            context = "\n".join(self.search(content))
            content = f"""Context:
{context}

Question: {content}"""
        super().add(role, content)

    def search(self, query: str) -> list[str]:
        """Return the top-k documents most similar to the query."""
        query_embed = self.embedding_model.embed(query)
        scores = [self._cosine(query_embed, embed) for embed in self.embeddings]
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        return [self.documents[index] for index in ranked[:3]]

    def _cosine(self, a: list[float], b: list[float]) -> float:
        """Calculate cosine similarity between two embeddings.."""
        return sum(x * y for x, y in zip(a, b))

Let’s put this to practice, starting with a set of documents that your TinyAgent has no direct access to. This is going to be a simple example, but imagine you have thousands of documents.

In [10]:
from illustrated_agents.chapters.ch4 import TinyAgent

# RAGMemory with external documents
documents = [
    "Sarah works as a marine biologist studying coral reefs.",
    "Sarah lives in Lisbon, Portugal.",
    "Sarah's favorite hobby is rock climbing.",
    "Sarah favorite animals are flamingos.",
    "Sarah speaks fluent Spanish and Portuguese.",
    "Ilse is a software engineer at a renewable energy startup.",
    "Ilse lives in Amsterdam, the Netherlands.",
    "Ilse plays the cello in a local string quartet.",
    "Ilse's favorite author is Brandon Sanderson.",
    "Ilse's favorite animals are dolphins.",
]
memory = RAGMemory(documents=documents, embedding_model=embedding_model)

# Create the Agent and run a query
agent = TinyAgent(llm=llm, memory=memory)
response = agent.run("What is Sarah's favorite animal?")
print(response)

Sarah's favorite animal is flamingos.

In [11]:
print(agent.memory.get_messages())

[
    {
        'role': 'user',
        'content': "Context:\nSarah favorite animals are flamingos.\nSarah's favorite hobby is rock 
climbing.\nIlse's favorite animals are dolphins.\n\nQuestion: What is Sarah's favorite animal?"
    },
    {'role': 'assistant', 'content': "Sarah's favorite animal is flamingos."}
]

Note how it retrieved the top 3 documents as the context? This is both the advantage and disadvantage of RAG. Although it minimizes the context that you have to pass to the model, there is no guarantee that the context will always be good enough. You could, for example, only accept documents that have a minimum degree of similarity rather than simply getting the top 3 irrespective of their absolute scores.

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how `Memory` could be added to your `TinyAgent`. There are now three main concepts in total (LLM, Memory, and TinyAgent):

In [1]:
from illustrated_agents.chapters.ch4 import what_we_built_lt; what_we_built_lt

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py                                                                                                    │
│ ├── llm.py        ← Updated (Add `EmbeddingModel` to support long-term memory.)                                 │
│ ├── memory.py     ← Updated (Add `LongTermMemory` to the TinyAgent in the form of RAG.)                         │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯